In [1]:
import pandas as pd
import os

sites = {
    "GSFC":   "NASA_Goddard",
    "HU_IRB": "HU_IRB",
    "SERC":   "SERC",
}

out_dir = "../../data/merged_aod_metero_2"
os.makedirs(out_dir, exist_ok=True)

for aod_key, meteo_key in sites.items():
    # --- Load AOD (UTC) ---
    aod = pd.read_csv(f"../../data/{aod_key}_AOD500_preprocessed.csv",
                      parse_dates=["datetime"])
    print(f"{aod_key} initial rows: {len(aod)}")

    # --- Step 1: Timezone conversion UTC → EST ---
    aod["datetime"] = aod["datetime"].dt.tz_localize("UTC").dt.tz_convert("America/New_York")
    aod["datetime"] = aod["datetime"].dt.tz_localize(None)

    # --- Step 2: Hourly average ---
    aod = aod.set_index("datetime").resample("1h").mean().reset_index()
    print(f"{aod_key} after 1H resample: {len(aod)} rows")

    aod = aod.sort_values("datetime")

    # --- Load meteo (local EST, hourly) ---
    meteo = pd.read_csv(f"../../data/metero/{meteo_key}_hourly_meteo.csv",
                        parse_dates=["DateTime"])
    meteo = meteo.sort_values("DateTime")

    # --- Step 3: Merge on exact hour ---
    merged = pd.merge(aod, meteo, left_on="datetime", right_on="DateTime", how="left")

    out_path = os.path.join(out_dir, f"{aod_key}_merged.csv")
    merged.to_csv(out_path, index=False)
    print(f"{aod_key}: {len(merged)} merged rows saved to {out_path}")
    print(f"  NaN meteo rows: {merged['temperature_2m'].isna().sum()} / {len(merged)}\n")


GSFC initial rows: 66337
GSFC after 1H resample: 44193 rows
GSFC: 45609 merged rows saved to ../../data/merged_aod_metero_2/GSFC_merged.csv
  NaN meteo rows: 329 / 45609

HU_IRB initial rows: 27972
HU_IRB after 1H resample: 33099 rows
HU_IRB: 34190 merged rows saved to ../../data/merged_aod_metero_2/HU_IRB_merged.csv
  NaN meteo rows: 0 / 34190

SERC initial rows: 39393
SERC after 1H resample: 38505 rows
SERC: 39753 merged rows saved to ../../data/merged_aod_metero_2/SERC_merged.csv
  NaN meteo rows: 0 / 39753



In [2]:
from IPython.display import display
df = pd.read_csv("../../data/merged_aod_metero_2/HU_IRB_merged.csv")
display(df.head())
print(df.shape)

df = pd.read_csv("../../data/HU_IRB_AOD500_preprocessed.csv")
display(df.head())
print(df.shape)


,datetime,AOD_500nm,DateTime,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,wind_direction_10m,Site
0,2021-03-01 13:00:00,0.163369,2021-03-01 13:00:00,8.7,82,0.9,28.1,310,HU IRB
1,2021-03-01 13:00:00,0.163369,2021-03-01 13:00:00,8.7,82,0.9,28.1,310,HU IRB
2,2021-03-01 14:00:00,0.144799,2021-03-01 14:00:00,8.0,75,0.0,22.7,306,HU IRB
3,2021-03-01 14:00:00,0.144799,2021-03-01 14:00:00,8.0,75,0.0,22.7,306,HU IRB
4,2021-03-01 15:00:00,0.167149,2021-03-01 15:00:00,9.1,60,0.0,23.1,301,HU IRB


(34190, 9)


,datetime,AOD_500nm
0,2021-03-01 18:56:43,0.163369
1,2021-03-01 19:01:43,0.144799
2,2021-03-01 20:16:42,0.170261
3,2021-03-01 20:21:42,0.179449
4,2021-03-01 20:30:52,0.152524


(27972, 2)
